In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1PvXvLRrVm4S2hHYmqS3pB0z_dRWKwFUk", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_00_intro.mp3"))

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Summary Memory: Compressing the Past

*Part 2 of the Vizuara series on Memory Architectures for LLM Applications*

**Estimated time: 55 minutes**

In the last notebook, we faced a hard tradeoff: the conversation buffer remembers everything but costs grow without bound, while the sliding window has fixed cost but drops critical information. What if there was a middle ground — what if we could *compress* old messages instead of deleting them? That is exactly what summary memory does. It uses the LLM itself to create a running summary of the conversation, preserving the gist while keeping token costs bounded. In this notebook, we will build a complete summary memory system, measure how much information survives compression, and discover exactly what gets lost.

In [ ]:
#@title 🎧 Listen: Why Matter
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_why_matter.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Think about how you remember a conversation from last week. You do not recall every single word. Instead, you remember the **key points**: "We talked about the project timeline, agreed on a March deadline, and decided to use Python." The specific phrasing is gone, but the important facts are preserved.

This is exactly what summary memory does for an LLM. Instead of keeping every message verbatim (expensive) or dropping old messages entirely (lossy), we ask the LLM to summarize the older parts of the conversation. The summary is short — maybe 200-500 tokens — but captures the essential facts.

The result is a two-part context:
1. **A compressed summary** of the older conversation (cheap, covers the big picture)
2. **The last few turns verbatim** (expensive per turn, but preserves recent detail)

This is like reading a Wikipedia article about a historical event (summary) and then reading today's newspaper for the latest updates (recent verbatim). You get both the big picture and the fine details.

In this notebook, we will:
- Build a **running summary system** that compresses conversation history using an LLM
- Implement two summarization strategies: **incremental** and **batch**
- Measure **information retention** — what survives compression and what does not
- Compare summary memory against buffer and sliding window on the same conversation
- Discover the **compression-information tradeoff** with real numbers

Let us begin.

In [ ]:
#@title 🎧 Listen: Intuition Analogy
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_intuition_analogy.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition

### The Compression Analogy

Imagine you are a court reporter transcribing a trial. After each day, the judge asks you to summarize the proceedings in one page. You write:

> *"The prosecution called two witnesses. Witness A testified that the defendant was at the location at 9:15 PM. Witness B provided financial records showing transactions on March 3. The defense cross-examined both witnesses on the reliability of their testimony."*

This one-page summary replaces perhaps 50 pages of verbatim testimony. The compression ratio is 50:1. But notice what was lost:
- The exact wording of questions and answers
- Emotional reactions and pauses
- Minor details that seemed unimportant at the time

And notice what was preserved:
- Who was involved (Witness A, Witness B)
- Key facts (9:15 PM, March 3, financial records)
- The structure of what happened (prosecution called witnesses, defense cross-examined)

This is the fundamental tradeoff of summary memory: **themes and key facts survive, but specific details and nuance are lost.**


In [ ]:
#@title 🎧 Listen: Intuition Arch
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_intuition_arch.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### How It Works Architecturally

The summary memory system maintains two components:

```
Context sent to LLM at each turn:
┌─────────────────────────────────┐
│  Summary of turns 1...(n-K)     │  ~300 tokens (compressed)
│  "User named Raj is building a  │
│   chatbot with $5K budget..."   │
├─────────────────────────────────┤
│  Recent turns (n-K+1)...n       │  ~2000 tokens (verbatim)
│  [Full messages as-is]          │
└─────────────────────────────────┘
```

The summary is re-generated periodically — either after every $N$ turns or when a token threshold is crossed. This keeps the old context compressed while preserving full fidelity for recent messages.

### The Key Question

Before we dive into code, think about this: if the user says "My budget is $5,000" in turn 2, and we summarize turns 1-10 at turn 15, will the summary preserve the exact number $5,000? Or will it say something vague like "the user has a moderate budget"?

This is the question we will answer empirically.

In [ ]:
#@title 🎧 Listen: Math Cost
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_math_cost.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Mathematics

### Token Cost of Summary Memory

Let $t_{\text{summary}}$ be the token count of the compressed summary, $K$ be the number of recent turns kept verbatim, and $\bar{t}$ be the average tokens per turn. The context cost is:

$$C_{\text{summary}}(n) = t_{\text{summary}} + \sum_{i=n-K+1}^{n} t_i \approx t_{\text{summary}} + K \cdot \bar{t}$$

This is bounded! No matter how long the conversation runs, we send roughly the same amount of context. Compare this to the buffer's $C_{\text{buffer}}(n) = n \cdot \bar{t}$.

**Worked example:** Suppose $t_{\text{summary}} = 300$ tokens, $K = 5$ recent turns, and $\bar{t} = 200$ tokens per turn. Then:

$$C_{\text{summary}} = 300 + 5 \times 200 = 1{,}300 \text{ tokens}$$

Whether the conversation has been 20 turns or 2,000 turns, we send approximately 1,300 tokens. The buffer at 2,000 turns would send $2{,}000 \times 200 = 400{,}000$ tokens. That is a **307x reduction**.

### Compression Ratio

We can define the compression ratio as:


In [ ]:
#@title 🎧 Listen: Math Cr Retention
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_math_cr_retention.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

$$\rho = \frac{t_{\text{summary}}}{\sum_{i=1}^{n-K} t_i}$$

This tells us how much the old messages were compressed. A lower $\rho$ means more compression.

**Worked example:** If we summarize 40 turns (each ~200 tokens = 8,000 tokens total) into a 300-token summary:

$$\rho = \frac{300}{8{,}000} = 0.0375 = 3.75\%$$

We compressed the old history to just 3.75% of its original size. That is aggressive compression — roughly 27:1. The question is: how much information survived?

### Information Retention Score

To measure what survives, we define an **information retention score**. Given a set of key facts $F = \{f_1, f_2, \ldots, f_m\}$ from the original conversation, we check how many appear in the summary:

$$\text{Retention}(F, S) = \frac{|\{f \in F : f \text{ appears in } S\}|}{|F|}$$

**Worked example:** Suppose the original conversation contained these facts:
- $f_1$: User name is Raj (appears in summary)
- $f_2$: Budget is $5,000 (appears as "~$5K budget")
- $f_3$: Deadline is March 15 (missing from summary)
- $f_4$: Team has 4 people (appears in summary)
- $f_5$: Frontend uses React (missing from summary)

$$\text{Retention} = \frac{3}{5} = 60\%$$

Three out of five facts survived. The deadline and tech stack detail were lost. This is typical of LLM summarization: names and round numbers survive, while specific dates and technical details are at risk.

### Setting Up Our Environment

In [ ]:
# Install dependencies
!pip install -q tiktoken numpy matplotlib

In [ ]:
#@title 🎧 Listen: Setup Complete
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_01_setup_complete.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Compare Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_21_compare_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
import tiktoken
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from typing import List, Dict, Tuple, Optional
import time
import json
import re

# Initialize tokenizer
ENCODER = tiktoken.encoding_for_model("gpt-4")

def count_tokens(text: str) -> int:
    """Count tokens in a string using tiktoken."""
    return len(ENCODER.encode(text))

print("Environment ready!")

In [ ]:
#@title 🎧 Listen: Sim Summ Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_sim_summ_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Let's Build It — Component by Component

### 4.1 A Simulated LLM Summarizer

In a production system, you would call an actual LLM (Gemini, Claude, GPT-4) to generate summaries. For this notebook, we will build a deterministic summarizer so our experiments are reproducible and free. The logic mimics what an LLM would do: extract key entities, facts, and themes.

In [ ]:
#@title 🎧 Listen: Sim Summ Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_sim_summ_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class SimulatedSummarizer:
    """A rule-based summarizer that mimics LLM behavior.

    Extracts key entities (names, numbers, dates) and
    produces a compressed summary. Deterministic and free.
    """

    def __init__(self):
        self.call_count = 0

    def summarize(self, text: str, max_length: int = 200) -> str:
        """Summarize a block of conversation text.

        Strategy:
        1. Extract named entities (names, numbers, dates)
        2. Identify key topics discussed
        3. Compress into a concise paragraph
        """
        self.call_count += 1

        # Extract key information
        entities = self._extract_entities(text)
        topics = self._extract_topics(text)
        facts = self._extract_facts(text)

        # Build summary
        parts = []
        if entities:
            parts.append(
                f"Participants/entities: {', '.join(entities[:5])}"
            )
        if facts:
            parts.append("Key facts: " + ". ".join(facts[:6]))
        if topics:
            parts.append(
                f"Topics discussed: {', '.join(topics[:5])}"
            )

        summary = ". ".join(parts)

        # Truncate to approximate max_length tokens
        words = summary.split()
        # Rough token estimate: 1 token ≈ 0.75 words
        max_words = int(max_length * 0.75)
        if len(words) > max_words:
            summary = " ".join(words[:max_words]) + "..."

        return summary

    def _extract_entities(self, text: str) -> List[str]:
        """Extract names, organizations, and proper nouns."""
        entities = set()
        # Match capitalized words (simple NER)
        for word in text.split():
            cleaned = word.strip(".,!?\"'()[]")
            if (cleaned and cleaned[0].isupper()
                    and len(cleaned) > 1
                    and cleaned not in (
                        "I", "The", "This", "That", "What",
                        "How", "When", "Where", "Yes", "No",
                        "Hi", "Hello", "Hey", "Sure", "Great",
                        "OK", "User", "Assistant")):
                entities.add(cleaned)
        return sorted(entities)

    def _extract_topics(self, text: str) -> List[str]:
        """Extract key topics from the conversation."""
        topic_keywords = {
            "budget": "budget planning",
            "deadline": "project timeline",
            "database": "database design",
            "api": "API development",
            "deploy": "deployment",
            "test": "testing",
            "python": "Python development",
            "frontend": "frontend development",
            "backend": "backend development",
            "machine learning": "machine learning",
            "authentication": "authentication",
            "performance": "performance optimization",
            "docker": "containerization",
            "security": "security",
        }
        found = []
        lower = text.lower()
        for keyword, topic in topic_keywords.items():
            if keyword in lower and topic not in found:
                found.append(topic)
        return found

    def _extract_facts(self, text: str) -> List[str]:
        """Extract specific facts (numbers, dates, etc)."""
        facts = []

        # Dollar amounts
        for match in re.findall(
            r'\$[\d,]+(?:\.\d{2})?', text
        ):
            facts.append(f"Budget/amount: {match}")

        # Dates
        for match in re.findall(
            r'(?:January|February|March|April|May|June|'
            r'July|August|September|October|November|'
            r'December)\s+\d{1,2}', text
        ):
            facts.append(f"Date mentioned: {match}")

        # Numbers with context
        for match in re.findall(
            r'(\d+)\s+(people|members|users|days|weeks|'
            r'months|hours|percent|%)', text
        ):
            facts.append(f"{match[0]} {match[1]}")

        # Email-like patterns
        for match in re.findall(
            r'[\w.]+@[\w.]+\.\w+', text
        ):
            facts.append(f"Contact: {match}")

        return facts

print("SimulatedSummarizer defined!")

Let us test the summarizer on a sample conversation:

In [ ]:
#@title 🎧 Listen: Sim Summ Test
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_sim_summ_test.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
summarizer = SimulatedSummarizer()

sample_conversation = """
User: My name is Raj and I am building a chatbot for
our e-commerce company. The budget is $5,000 and the
deadline is March 15.
Assistant: Great, Raj! With a $5,000 budget and March 15
deadline, we need to plan carefully. What tech stack
are you considering?
User: The team has 4 people: me, Sarah on frontend,
Alex on backend, and Priya on ML. We want to use
Python with FastAPI.
Assistant: Excellent team composition. Python with FastAPI
is a solid choice for the backend. Sarah can use React
for the frontend, and Priya can integrate ML models.
User: We need to handle authentication and user sessions.
Assistant: For authentication, I recommend using OAuth2
with JWT tokens. FastAPI has great support for this.
"""

summary = summarizer.summarize(sample_conversation)
original_tokens = count_tokens(sample_conversation)
summary_tokens = count_tokens(summary)

print(f"Original: {original_tokens} tokens")
print(f"Summary:  {summary_tokens} tokens")
print(f"Compression ratio: {summary_tokens/original_tokens:.1%}")
print(f"\n--- Summary ---")
print(summary)

In [ ]:
#@title 🎧 Listen: Summ Mem Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_summ_mem_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.2 The Summary Memory Class

Now let us build the actual memory system. It maintains a running summary of older messages and keeps the last $K$ turns verbatim.

In [ ]:
#@title 🎧 Listen: Summ Mem Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_summ_mem_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class SummaryMemory:
    """Memory system that compresses old messages into
    a running summary while keeping recent turns verbatim.

    Architecture:
        Context = [Summary of old turns] + [Last K turns]

    The summary is updated every `summarize_every` turns,
    compressing all messages outside the recent window.
    """

    def __init__(
        self,
        summarizer,
        recent_turns: int = 5,
        summarize_every: int = 5,
        system_prompt: str = "",
    ):
        self.summarizer = summarizer
        self.recent_turns = recent_turns
        self.summarize_every = summarize_every
        self.system_prompt = system_prompt
        self.system_tokens = count_tokens(system_prompt)

        # State
        self.all_messages: List[Dict] = []
        self.summary: str = ""
        self.summary_tokens: int = 0
        self.turns_since_summary: int = 0

        # Tracking
        self.total_token_history: List[int] = []
        self.context_token_history: List[int] = []
        self.summary_history: List[str] = []
        self.compression_ratios: List[float] = []

    def add_message(self, role: str, content: str):
        """Add a message and trigger summarization if needed."""
        tokens = count_tokens(content)
        self.all_messages.append({
            "role": role,
            "content": content,
            "tokens": tokens,
        })

        # Track total tokens (everything ever said)
        total = sum(m["tokens"] for m in self.all_messages)
        self.total_token_history.append(total)

        # Count turns (a turn = user + assistant pair)
        if role == "assistant":
            self.turns_since_summary += 1

            # Trigger summarization if threshold reached
            if (self.turns_since_summary >= self.summarize_every
                    and len(self.all_messages) > self.recent_turns * 2):
                self._update_summary()
                self.turns_since_summary = 0

        # Track context size
        ctx_tokens = self.get_context_tokens()
        self.context_token_history.append(ctx_tokens)

    def _update_summary(self):
        """Re-summarize all messages outside recent window."""
        recent_count = self.recent_turns * 2
        if len(self.all_messages) <= recent_count:
            return

        old_messages = self.all_messages[:-recent_count]
        old_text = "\n".join(
            f"{m['role']}: {m['content']}"
            for m in old_messages
        )
        old_tokens = sum(m["tokens"] for m in old_messages)

        # Generate summary
        self.summary = self.summarizer.summarize(old_text)
        self.summary_tokens = count_tokens(self.summary)

        # Track compression
        if old_tokens > 0:
            ratio = self.summary_tokens / old_tokens
            self.compression_ratios.append(ratio)

        self.summary_history.append(self.summary)

    def get_context(self) -> str:
        """Build the context string for the LLM."""
        parts = []
        if self.system_prompt:
            parts.append(f"System: {self.system_prompt}")
        if self.summary:
            parts.append(
                f"Summary of earlier conversation:\n{self.summary}"
            )

        # Recent messages (verbatim)
        recent_count = self.recent_turns * 2
        recent = self.all_messages[-recent_count:]
        for m in recent:
            parts.append(f"{m['role']}: {m['content']}")

        return "\n\n".join(parts)

    def get_context_tokens(self) -> int:
        """Token count of the context we would send."""
        total = self.system_tokens + self.summary_tokens
        recent_count = self.recent_turns * 2
        recent = self.all_messages[-recent_count:]
        total += sum(m["tokens"] for m in recent)
        return total

    def get_stats(self) -> Dict:
        """Return current memory statistics."""
        return {
            "total_messages": len(self.all_messages),
            "total_tokens": sum(
                m["tokens"] for m in self.all_messages
            ),
            "summary_tokens": self.summary_tokens,
            "context_tokens": self.get_context_tokens(),
            "compression_ratio": (
                self.compression_ratios[-1]
                if self.compression_ratios
                else None
            ),
            "summarizer_calls": self.summarizer.call_count,
        }

print("SummaryMemory defined!")

Let us test it with a multi-turn conversation:

In [ ]:
#@title 🎧 Listen: Summ Mem Test
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_summ_mem_test.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
mem = SummaryMemory(
    summarizer=SimulatedSummarizer(),
    recent_turns=3,
    summarize_every=3,
    system_prompt="You are a project planning assistant.",
)

# Simulate 15 turns of conversation
conversation_turns = [
    ("user", "My name is Raj. I am building a chatbot for "
     "e-commerce customer support."),
    ("assistant", "Hello Raj! An e-commerce chatbot is a "
     "great project. What is your budget and timeline?"),
    ("user", "The budget is $5,000 and the deadline is "
     "March 15. The team has 4 people."),
    ("assistant", "With $5,000 and a March 15 deadline for "
     "a team of 4, we should prioritize an MVP first."),
    ("user", "Sarah handles frontend, Alex does backend, "
     "and Priya works on the ML models."),
    ("assistant", "Great team allocation. Sarah on frontend "
     "with React, Alex on FastAPI backend, and Priya on ML."),
    ("user", "For the database, I am thinking PostgreSQL "
     "with pgvector for embeddings."),
    ("assistant", "PostgreSQL with pgvector is excellent. "
     "It handles both relational data and vector similarity "
     "search in one system."),
    ("user", "We also need OAuth2 authentication and "
     "rate limiting for the API."),
    ("assistant", "For OAuth2, use FastAPI's built-in "
     "security. For rate limiting, consider SlowAPI or "
     "a reverse proxy like Nginx."),
    ("user", "What about monitoring and logging?"),
    ("assistant", "Use Prometheus with Grafana for metrics, "
     "and structured logging with loguru. Set up alerts "
     "for error rates and latency."),
    ("user", "The chatbot needs to handle returns, order "
     "tracking, and FAQ questions."),
    ("assistant", "Three main features: returns processing, "
     "order tracking via API, and FAQ with RAG-based "
     "retrieval. Prioritize order tracking first."),
    ("user", "How should we handle the deployment?"),
    ("assistant", "Deploy with Docker on AWS ECS or "
     "Google Cloud Run. Use GitHub Actions for CI/CD "
     "and Terraform for infrastructure as code."),
    ("user", "Let us plan the sprint schedule."),
    ("assistant", "With 4 weeks until March 15, I suggest: "
     "Week 1 — core backend and database. Week 2 — "
     "chatbot logic and ML pipeline. Week 3 — frontend "
     "and integration. Week 4 — testing and deployment."),
    ("user", "What about the ML model for the chatbot?"),
    ("assistant", "Use a fine-tuned LLM via API (GPT-4 or "
     "Claude) with RAG for grounding. Priya can build "
     "the retrieval pipeline using your pgvector setup."),
]

print("Feeding conversation into SummaryMemory...")
print("=" * 55)

for i, (role, content) in enumerate(conversation_turns):
    mem.add_message(role, content)
    if role == "assistant":
        turn_num = (i + 1) // 2
        stats = mem.get_stats()
        has_summary = "YES" if mem.summary else "no"
        print(
            f"  Turn {turn_num:>2}: "
            f"total={stats['total_tokens']:>5} tok, "
            f"context={stats['context_tokens']:>4} tok, "
            f"summary={has_summary}"
        )

print(f"\n--- Final Summary ---")
print(mem.summary)
print(f"\nSummary tokens: {mem.summary_tokens}")
print(f"Summarizer was called {mem.summarizer.call_count} times")

Notice what happened. As the conversation grew, the summary was updated periodically. The context token count stays bounded around the same size regardless of how many total tokens have accumulated. But the real question is: what information survived the compression? Let us measure that next.

In [ ]:
#@title 🎧 Listen: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Your Turn

### TODO 1: Implement an Information Retention Scorer

Given a list of key facts and a summary text, compute what fraction of the facts are preserved in the summary. This is how we measure summary quality.

In [ ]:
#@title 🎧 Listen: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def score_information_retention(
    key_facts: List[Dict[str, str]],
    summary: str,
    recent_context: str = "",
) -> Dict:
    """Score how many key facts are retained in the
    summary + recent context.

    Args:
        key_facts: List of dicts, each with:
            - 'fact': description of the fact
            - 'keywords': list of keywords that indicate
              the fact is present (ANY keyword match = found)
            - 'category': type of fact ('name', 'number',
              'date', 'preference', 'technical')
        summary: The compressed summary text
        recent_context: The recent verbatim messages

    Returns:
        Dict with keys:
            'total_facts': number of facts checked
            'retained_count': number of facts found
            'retention_rate': retained_count / total_facts
            'per_category': dict mapping category to
                {'total': int, 'retained': int, 'rate': float}
            'lost_facts': list of fact descriptions that
                were NOT found

    Steps:
        1. Combine summary and recent_context into one
           search text (lowercase)
        2. For each fact, check if ANY of its keywords
           appear in the search text
        3. Track retention per category
        4. Build and return the results dict

    Example:
        >>> facts = [
        ...   {'fact': 'Name is Raj',
        ...    'keywords': ['raj'],
        ...    'category': 'name'},
        ...   {'fact': 'Budget is $5,000',
        ...    'keywords': ['5,000', '5000', '5k'],
        ...    'category': 'number'},
        ... ]
        >>> result = score_information_retention(
        ...     facts, "User Raj has a moderate budget"
        ... )
        >>> result['retention_rate']
        0.5  # Found 'raj' but not '5,000'
    """
    # ---- YOUR CODE HERE ---- #

    results = {
        "total_facts": len(key_facts),
        "retained_count": 0,
        "retention_rate": 0.0,
        "per_category": {},
        "lost_facts": [],
    }

    # TODO: Combine summary + recent_context (lowercase)
    # TODO: For each fact, check keyword match
    # TODO: Track per-category stats
    # TODO: Compute overall retention rate

    return results

    # ---- END YOUR CODE ---- #

In [ ]:
#@title 🎧 Listen: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 2: Implement an Incremental Summarizer

Our current system re-summarizes all old messages from scratch every time. This is wasteful — if we already have a summary, we should just *update* it with the newly-old messages. Implement `IncrementalSummaryMemory` that updates the existing summary rather than regenerating it.

In [ ]:
#@title 🎧 Listen: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class IncrementalSummaryMemory:
    """Summary memory that updates incrementally.

    Instead of re-summarizing all old messages, this
    takes the existing summary and the newly-old messages,
    and asks the summarizer to produce an updated summary.

    This is more efficient because the summarizer processes
    less text on each update.

    Args:
        summarizer: A summarizer with a .summarize() method
        recent_turns: Number of recent turns to keep verbatim
        update_every: Trigger summary update every N turns
    """

    def __init__(
        self,
        summarizer,
        recent_turns: int = 5,
        update_every: int = 3,
    ):
        self.summarizer = summarizer
        self.recent_turns = recent_turns
        self.update_every = update_every
        self.all_messages: List[Dict] = []
        self.summary: str = ""
        self.summary_tokens: int = 0
        self.turns_since_update: int = 0
        self.last_summarized_idx: int = 0
        self.update_count: int = 0
        self.context_history: List[int] = []

    def add_message(self, role: str, content: str):
        """Add a message and trigger incremental update
        if needed.

        Requirements:
            1. Add the message to self.all_messages with
               role, content, and tokens keys
            2. If role is 'assistant', increment
               turns_since_update
            3. If turns_since_update >= update_every
               AND there are messages outside the recent
               window that have not been summarized:
               a. Get the newly-old messages (between
                  last_summarized_idx and the start of
                  the recent window)
               b. Build input text as:
                  "Previous summary: {self.summary}\n\n
                   New messages:\n{newly_old_text}"
               c. Call self.summarizer.summarize(input_text)
               d. Update self.summary and self.summary_tokens
               e. Update self.last_summarized_idx
               f. Reset turns_since_update
               g. Increment update_count
            4. Track context token count in context_history

        Hints:
            - Recent window starts at index
              max(0, len(self.all_messages) - recent_turns * 2)
            - Newly-old messages are from last_summarized_idx
              to the start of recent window
        """
        # ---- YOUR CODE HERE ---- #

        tokens = count_tokens(content)
        self.all_messages.append({
            "role": role,
            "content": content,
            "tokens": tokens,
        })

        # TODO: Check if we should update the summary
        # TODO: If so, build incremental input and summarize
        # TODO: Track context tokens

        # ---- END YOUR CODE ---- #

    def get_context_tokens(self) -> int:
        """Compute tokens in current context."""
        recent_start = max(
            0, len(self.all_messages) - self.recent_turns * 2
        )
        recent = self.all_messages[recent_start:]
        return self.summary_tokens + sum(
            m["tokens"] for m in recent
        )

In [ ]:
#@title 🎧 Listen: Todo1 Verify
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_todo1_verify.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Verification Cell for TODO 1

In [ ]:
# --- Test score_information_retention ---
test_facts = [
    {"fact": "User name is Raj",
     "keywords": ["raj"],
     "category": "name"},
    {"fact": "Budget is $5,000",
     "keywords": ["5,000", "5000", "$5k", "5k"],
     "category": "number"},
    {"fact": "Deadline is March 15",
     "keywords": ["march 15", "march15"],
     "category": "date"},
    {"fact": "Team has 4 people",
     "keywords": ["4 people", "four people", "team of 4"],
     "category": "number"},
    {"fact": "Using Python with FastAPI",
     "keywords": ["fastapi", "fast api"],
     "category": "technical"},
    {"fact": "Sarah does frontend",
     "keywords": ["sarah"],
     "category": "name"},
    {"fact": "PostgreSQL for database",
     "keywords": ["postgresql", "postgres"],
     "category": "technical"},
]

test_summary = (
    "Participants/entities: Raj, Sarah, Alex, Priya. "
    "Key facts: Budget/amount: $5,000. 4 people on the team. "
    "Topics discussed: Python development, database design."
)

result = score_information_retention(test_facts, test_summary)

assert result["total_facts"] == 7
assert result["retained_count"] >= 3, \
    f"Expected >= 3 retained, got {result['retained_count']}"
assert 0 <= result["retention_rate"] <= 1
assert "name" in result["per_category"]
assert len(result["lost_facts"]) == (
    result["total_facts"] - result["retained_count"]
)

print(f"Retention: {result['retained_count']}/{result['total_facts']} "
      f"({result['retention_rate']:.0%})")
print(f"Lost facts: {result['lost_facts']}")
print("\nPer category:")
for cat, stats in result["per_category"].items():
    print(f"  {cat}: {stats['retained']}/{stats['total']} "
          f"({stats['rate']:.0%})")

print("\nAll retention scorer tests passed!")

In [ ]:
#@title 🎧 Listen: Todo2 Verify
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_todo2_verify.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Verification Cell for TODO 2

In [ ]:
# --- Test IncrementalSummaryMemory ---
inc_mem = IncrementalSummaryMemory(
    summarizer=SimulatedSummarizer(),
    recent_turns=2,
    update_every=2,
)

test_msgs = [
    ("user", "My name is Raj and I work at Google."),
    ("assistant", "Hello Raj! How can I help you today?"),
    ("user", "I need help with a Python project."),
    ("assistant", "Sure! What kind of Python project?"),
    ("user", "A web scraper for product prices."),
    ("assistant", "I recommend using BeautifulSoup or Scrapy."),
    ("user", "The budget is $2,000 for this project."),
    ("assistant", "With $2,000, Scrapy with a cloud deployment is good."),
]

for role, content in test_msgs:
    inc_mem.add_message(role, content)

assert inc_mem.update_count >= 1, \
    f"Expected at least 1 update, got {inc_mem.update_count}"
assert inc_mem.summary != "", "Summary should not be empty"
assert inc_mem.summary_tokens > 0
assert len(inc_mem.context_history) == len(test_msgs)

print(f"Updates triggered: {inc_mem.update_count}")
print(f"Summary: {inc_mem.summary[:100]}...")
print(f"Summary tokens: {inc_mem.summary_tokens}")
print(f"Context tokens: {inc_mem.get_context_tokens()}")
print("\nAll IncrementalSummaryMemory tests passed!")

In [ ]:
#@title 🎧 Listen: Solutions
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_19_solutions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

<details>
<summary><b>Solution: score_information_retention</b></summary>

In [ ]:
def score_information_retention(key_facts, summary, recent_context=""):
    search_text = (summary + " " + recent_context).lower()

    per_category = {}
    retained_count = 0
    lost_facts = []

    for fact_info in key_facts:
        cat = fact_info["category"]
        if cat not in per_category:
            per_category[cat] = {"total": 0, "retained": 0, "rate": 0.0}
        per_category[cat]["total"] += 1

        found = any(kw.lower() in search_text for kw in fact_info["keywords"])
        if found:
            retained_count += 1
            per_category[cat]["retained"] += 1
        else:
            lost_facts.append(fact_info["fact"])

    for cat in per_category:
        t = per_category[cat]["total"]
        r = per_category[cat]["retained"]
        per_category[cat]["rate"] = r / t if t > 0 else 0.0

    return {
        "total_facts": len(key_facts),
        "retained_count": retained_count,
        "retention_rate": retained_count / len(key_facts) if key_facts else 0,
        "per_category": per_category,
        "lost_facts": lost_facts,
    }

</details>

<details>
<summary><b>Solution: IncrementalSummaryMemory.add_message</b></summary>

In [ ]:
def add_message(self, role, content):
    tokens = count_tokens(content)
    self.all_messages.append({
        "role": role, "content": content, "tokens": tokens,
    })

    if role == "assistant":
        self.turns_since_update += 1

        recent_start = max(0, len(self.all_messages) - self.recent_turns * 2)

        if (self.turns_since_update >= self.update_every
                and self.last_summarized_idx < recent_start):
            newly_old = self.all_messages[self.last_summarized_idx:recent_start]
            newly_old_text = "\n".join(
                f"{m['role']}: {m['content']}" for m in newly_old
            )
            input_text = ""
            if self.summary:
                input_text = f"Previous summary: {self.summary}\n\nNew messages:\n{newly_old_text}"
            else:
                input_text = newly_old_text

            self.summary = self.summarizer.summarize(input_text)
            self.summary_tokens = count_tokens(self.summary)
            self.last_summarized_idx = recent_start
            self.turns_since_update = 0
            self.update_count += 1

    self.context_history.append(self.get_context_tokens())

</details>

In [ ]:
# --- Load solutions ---
def score_information_retention(key_facts, summary, recent_context=""):
    search_text = (summary + " " + recent_context).lower()
    per_category = {}
    retained_count = 0
    lost_facts = []

    for fact_info in key_facts:
        cat = fact_info["category"]
        if cat not in per_category:
            per_category[cat] = {"total": 0, "retained": 0, "rate": 0.0}
        per_category[cat]["total"] += 1
        found = any(kw.lower() in search_text for kw in fact_info["keywords"])
        if found:
            retained_count += 1
            per_category[cat]["retained"] += 1
        else:
            lost_facts.append(fact_info["fact"])

    for cat in per_category:
        t = per_category[cat]["total"]
        r = per_category[cat]["retained"]
        per_category[cat]["rate"] = r / t if t > 0 else 0.0

    return {
        "total_facts": len(key_facts),
        "retained_count": retained_count,
        "retention_rate": retained_count / len(key_facts) if key_facts else 0,
        "per_category": per_category,
        "lost_facts": lost_facts,
    }

print("Solutions loaded!")

In [ ]:
#@title 🎧 Listen: Compare Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_20_compare_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Putting It All Together

Let us now run a head-to-head comparison of buffer, sliding window, and summary memory on the same conversation. We will measure both token cost and information retention.

In [ ]:
from collections import deque

class SlidingWindowMemory:
    """Sliding window for comparison."""
    def __init__(self, window_size=5, system_prompt=""):
        self.window_size = window_size
        self.messages = []
        self.system_tokens = count_tokens(system_prompt)
        self.context_history = []

    def add_message(self, role, content):
        tokens = count_tokens(content)
        self.messages.append({
            "role": role, "content": content, "tokens": tokens,
        })
        ctx = self._get_window()
        ctx_tokens = self.system_tokens + sum(
            m["tokens"] for m in ctx
        )
        self.context_history.append(ctx_tokens)

    def _get_window(self):
        max_msgs = self.window_size * 2
        return self.messages[-max_msgs:]

    def get_context(self):
        return "\n".join(
            f"{m['role']}: {m['content']}"
            for m in self._get_window()
        )

class BufferMemory:
    """Buffer for comparison."""
    def __init__(self, system_prompt=""):
        self.messages = []
        self.system_tokens = count_tokens(system_prompt)
        self.context_history = []

    def add_message(self, role, content):
        tokens = count_tokens(content)
        self.messages.append({
            "role": role, "content": content, "tokens": tokens,
        })
        total = self.system_tokens + sum(
            m["tokens"] for m in self.messages
        )
        self.context_history.append(total)

    def get_context(self):
        return "\n".join(
            f"{m['role']}: {m['content']}"
            for m in self.messages
        )

print("Comparison classes ready!")

In [ ]:
#@title 🎧 Listen: Run Comparison
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_22_run_comparison.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Final Demo Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_30_final_demo_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Run the same conversation through all three systems
sys_prompt = "You are a project planning assistant."

buf = BufferMemory(system_prompt=sys_prompt)
win = SlidingWindowMemory(window_size=5, system_prompt=sys_prompt)
summ = SummaryMemory(
    summarizer=SimulatedSummarizer(),
    recent_turns=5,
    summarize_every=5,
    system_prompt=sys_prompt,
)

# Feed the same conversation
for role, content in conversation_turns:
    buf.add_message(role, content)
    win.add_message(role, content)
    summ.add_message(role, content)

# Compare final state
print("=" * 60)
print("HEAD-TO-HEAD COMPARISON (10 turns)")
print("=" * 60)

buf_ctx = buf.context_history[-1]
win_ctx = win.context_history[-1]
sum_ctx = summ.context_token_history[-1]

print(f"\n{'Strategy':<25} {'Context Tokens':>15}")
print("-" * 42)
print(f"{'Buffer (full):':<25} {buf_ctx:>15,}")
print(f"{'Window (K=5):':<25} {win_ctx:>15,}")
print(f"{'Summary (K=5):':<25} {sum_ctx:>15,}")

print(f"\nSavings vs Buffer:")
print(f"  Window: {(1 - win_ctx/buf_ctx)*100:.1f}% reduction")
print(f"  Summary: {(1 - sum_ctx/buf_ctx)*100:.1f}% reduction")

In [ ]:
#@title 🎧 Listen: Viz1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_23_viz1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Training and Results

### Visualization 1: Token Cost Comparison Over Time

In [ ]:
#@title 🎧 Listen: Viz1 Plot
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_24_viz1_plot.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Get context history at assistant messages only
# (every other entry)
buf_ctx_turns = buf.context_history[1::2]
win_ctx_turns = win.context_history[1::2]
sum_ctx_turns = summ.context_token_history[1::2]

n = min(len(buf_ctx_turns), len(win_ctx_turns),
        len(sum_ctx_turns))
turns_x = list(range(1, n + 1))

# Left: Token cost over time
ax1 = axes[0]
ax1.plot(turns_x, buf_ctx_turns[:n], color="#EF4444",
         linewidth=2.5, label="Buffer (full)")
ax1.plot(turns_x, win_ctx_turns[:n], color="#10B981",
         linewidth=2.5, linestyle="--",
         label="Window K=5")
ax1.plot(turns_x, sum_ctx_turns[:n], color="#3B82F6",
         linewidth=2.5, linestyle="-.",
         label="Summary K=5")

ax1.set_xlabel("Conversation Turn", fontsize=12)
ax1.set_ylabel("Context Tokens Sent to LLM", fontsize=12)
ax1.set_title("Token Cost Over Time",
              fontsize=14, fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Right: Cumulative cost (total tokens sent across all turns)
ax2 = axes[1]
buf_cumul = np.cumsum(buf_ctx_turns[:n])
win_cumul = np.cumsum(win_ctx_turns[:n])
sum_cumul = np.cumsum(sum_ctx_turns[:n])

ax2.fill_between(turns_x, buf_cumul, alpha=0.2,
                 color="#EF4444")
ax2.plot(turns_x, buf_cumul, color="#EF4444",
         linewidth=2.5, label="Buffer")
ax2.fill_between(turns_x, sum_cumul, alpha=0.2,
                 color="#3B82F6")
ax2.plot(turns_x, sum_cumul, color="#3B82F6",
         linewidth=2.5, label="Summary")
ax2.fill_between(turns_x, win_cumul, alpha=0.2,
                 color="#10B981")
ax2.plot(turns_x, win_cumul, color="#10B981",
         linewidth=2.5, label="Window")

ax2.set_xlabel("Conversation Turn", fontsize=12)
ax2.set_ylabel("Cumulative Tokens Sent (All Turns)",
               fontsize=12)
ax2.set_title("Total API Token Usage",
              fontsize=14, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("summary_cost_comparison.png", dpi=150,
            bbox_inches="tight")
plt.show()

In [ ]:
#@title 🎧 Listen: Viz2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_25_viz2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Visualization 2: Information Retention by Fact Category

Let us measure what each system remembers about the conversation using our retention scorer.

In [ ]:
#@title 🎧 Listen: Viz2 Plot
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_26_viz2_plot.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Define the key facts from our conversation
key_facts = [
    {"fact": "User name is Raj",
     "keywords": ["raj"], "category": "name"},
    {"fact": "Budget is $5,000",
     "keywords": ["5,000", "5000", "$5k"],
     "category": "number"},
    {"fact": "Deadline is March 15",
     "keywords": ["march 15"], "category": "date"},
    {"fact": "Team has 4 people",
     "keywords": ["4 people", "team of 4", "4,"],
     "category": "number"},
    {"fact": "Sarah does frontend",
     "keywords": ["sarah"], "category": "name"},
    {"fact": "Alex does backend",
     "keywords": ["alex"], "category": "name"},
    {"fact": "Priya does ML",
     "keywords": ["priya"], "category": "name"},
    {"fact": "Using FastAPI",
     "keywords": ["fastapi"], "category": "technical"},
    {"fact": "PostgreSQL database",
     "keywords": ["postgresql", "postgres", "pgvector"],
     "category": "technical"},
    {"fact": "OAuth2 authentication",
     "keywords": ["oauth"], "category": "technical"},
    {"fact": "Docker deployment",
     "keywords": ["docker"], "category": "technical"},
    {"fact": "Sprint is 4 weeks",
     "keywords": ["4 weeks", "week 1", "week 2"],
     "category": "plan"},
]

# Score each system
buf_score = score_information_retention(
    key_facts, buf.get_context()
)
win_score = score_information_retention(
    key_facts, win.get_context()
)
sum_score = score_information_retention(
    key_facts, summ.get_context()
)

print("Information Retention Comparison")
print("=" * 55)
print(f"{'System':<20} {'Retained':>10} {'Rate':>10}")
print("-" * 42)
print(f"{'Buffer:':<20} "
      f"{buf_score['retained_count']}/{buf_score['total_facts']:>7} "
      f"{buf_score['retention_rate']:>9.0%}")
print(f"{'Window K=5:':<20} "
      f"{win_score['retained_count']}/{win_score['total_facts']:>7} "
      f"{win_score['retention_rate']:>9.0%}")
print(f"{'Summary K=5:':<20} "
      f"{sum_score['retained_count']}/{sum_score['total_facts']:>7} "
      f"{sum_score['retention_rate']:>9.0%}")

print(f"\nFacts lost by Window: {win_score['lost_facts']}")
print(f"Facts lost by Summary: {sum_score['lost_facts']}")

In [ ]:
# Visualize retention by category
categories = sorted(set(
    f["category"] for f in key_facts
))

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(categories))
width = 0.25

for i, (score, label, color) in enumerate([
    (buf_score, "Buffer", "#EF4444"),
    (win_score, "Window K=5", "#10B981"),
    (sum_score, "Summary K=5", "#3B82F6"),
]):
    rates = []
    for cat in categories:
        if cat in score["per_category"]:
            rates.append(score["per_category"][cat]["rate"])
        else:
            rates.append(0)
    ax.bar(x + i * width, rates, width,
           label=label, color=color, alpha=0.8)

ax.set_xticks(x + width)
ax.set_xticklabels([c.capitalize() for c in categories],
                    fontsize=11)
ax.set_ylabel("Retention Rate", fontsize=12)
ax.set_title("Information Retention by Fact Category",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.set_ylim(0, 1.15)
ax.axhline(y=1.0, color="gray", linestyle=":",
           alpha=0.3)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("retention_by_category.png", dpi=150,
            bbox_inches="tight")
plt.show()

In [ ]:
#@title 🎧 Listen: Viz3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_27_viz3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Visualization 3: The Compression-Retention Tradeoff

Let us explore how different summary lengths affect information retention.

In [ ]:
#@title 🎧 Listen: Viz3 Plot
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_28_viz3_plot.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Sweep over different summary max lengths
summary_lengths = [50, 100, 150, 200, 300, 500, 800]
retention_rates = []
compression_ratios = []

# Build the old conversation text
old_text = "\n".join(
    f"{m['role']}: {m['content']}"
    for m in conversation_turns[:-10]  # Exclude last 5 turns
)
old_tokens = count_tokens(old_text)

sweep_summarizer = SimulatedSummarizer()

for max_len in summary_lengths:
    summary = sweep_summarizer.summarize(old_text, max_length=max_len)
    s_tokens = count_tokens(summary)

    # Score retention
    score = score_information_retention(key_facts, summary)
    retention_rates.append(score["retention_rate"])
    compression_ratios.append(
        s_tokens / old_tokens if old_tokens > 0 else 0
    )

fig, ax1 = plt.subplots(figsize=(12, 6))
ax2 = ax1.twinx()

ax1.plot(summary_lengths, retention_rates, "o-",
         color="#3B82F6", linewidth=2.5, markersize=8,
         label="Information Retention")
ax2.plot(summary_lengths, compression_ratios, "s--",
         color="#EF4444", linewidth=2.5, markersize=8,
         label="Compression Ratio")

ax1.set_xlabel("Summary Max Length (tokens)", fontsize=12)
ax1.set_ylabel("Information Retention Rate",
               fontsize=12, color="#3B82F6")
ax2.set_ylabel("Compression Ratio (summary / original)",
               fontsize=12, color="#EF4444")
ax1.set_title("The Compression-Retention Tradeoff",
              fontsize=14, fontweight="bold")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           fontsize=10, loc="center right")

ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("compression_tradeoff.png", dpi=150,
            bbox_inches="tight")
plt.show()

print("As summary length increases, retention improves")
print("but compression gets worse. The sweet spot is")
print("typically 200-300 tokens for a 10-turn conversation.")

In [ ]:
#@title 🎧 Listen: Final Demo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_29_final_demo_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Final Output

Let us run a complete demonstration showing summary memory handling the "callback" scenario — where the user asks about something from early in the conversation.

In [ ]:
print("=" * 60)
print("  FINAL DEMO: SUMMARY MEMORY HANDLES CALLBACKS")
print("=" * 60)

# Create a fresh summary memory
demo_mem = SummaryMemory(
    summarizer=SimulatedSummarizer(),
    recent_turns=3,
    summarize_every=3,
    system_prompt="You are a project assistant.",
)

# Phase 1: Key facts (turns 1-3)
print("\n--- Phase 1: Establishing Key Facts ---")
phase1 = [
    ("user", "I am Raj, budget is $5,000, deadline March 15."),
    ("assistant", "Got it Raj! $5,000 budget, March 15 deadline."),
    ("user", "Team: Sarah (frontend), Alex (backend), Priya (ML)."),
    ("assistant", "Great team of 4. What is the project about?"),
    ("user", "E-commerce chatbot using Python and FastAPI."),
    ("assistant", "Python + FastAPI is a solid choice for this."),
]

for role, content in phase1:
    demo_mem.add_message(role, content)
    if role == "user":
        print(f"  User: {content[:60]}...")

# Phase 2: Topic drift (turns 4-8)
print("\n--- Phase 2: Topic Drift ---")
phase2 = [
    ("user", "What about CI/CD pipelines?"),
    ("assistant", "Use GitHub Actions for automated testing and deployment."),
    ("user", "And database options?"),
    ("assistant", "PostgreSQL with pgvector for embeddings."),
    ("user", "How about containerization?"),
    ("assistant", "Docker with Docker Compose locally, K8s for prod."),
    ("user", "What monitoring tools?"),
    ("assistant", "Prometheus and Grafana for metrics and dashboards."),
    ("user", "Security considerations?"),
    ("assistant", "OAuth2, rate limiting, input validation, HTTPS."),
]

for role, content in phase2:
    demo_mem.add_message(role, content)
    if role == "user":
        print(f"  User: {content[:60]}...")

# Phase 3: The callback
print("\n--- Phase 3: The Callback ---")
callback = "Remind me — what was our budget and who is on the team?"
demo_mem.add_message("user", callback)
print(f"  User: {callback}")

print("\n--- Context Available to LLM ---")
print(f"\n  [SUMMARY] ({demo_mem.summary_tokens} tokens):")
if demo_mem.summary:
    for line in demo_mem.summary.split(". "):
        if line.strip():
            print(f"    {line.strip()}")

print(f"\n  [RECENT MESSAGES]:")
recent_count = demo_mem.recent_turns * 2
recent = demo_mem.all_messages[-recent_count:]
for m in recent:
    print(f"    [{m['role']:>9}] {m['content'][:55]}...")

# Check what is preserved
stats = demo_mem.get_stats()
context = demo_mem.get_context()
budget_found = "5,000" in context or "5000" in context
raj_found = "raj" in context.lower()
sarah_found = "sarah" in context.lower()

print(f"\n--- Fact Check ---")
print(f"  Budget ($5,000) in context: {'YES' if budget_found else 'NO'}")
print(f"  Name (Raj) in context: {'YES' if raj_found else 'NO'}")
print(f"  Team (Sarah) in context: {'YES' if sarah_found else 'NO'}")
print(f"\n  Context tokens: {stats['context_tokens']}")
print(f"  Total conversation tokens: {stats['total_tokens']}")
print(f"  Compression: {stats['context_tokens']/stats['total_tokens']:.1%} "
      f"of original")

print(f"\n{'=' * 60}")
print(f"Summary memory preserves key facts from early turns")
print(f"that the sliding window would have lost entirely.")
print(f"{'=' * 60}")

In [ ]:
#@title 🎧 Listen: Reflection Learned
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_31_reflection_learned.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Reflection and Next Steps

### What Did We Learn?

1. **Summary memory gives us bounded cost AND information retention.** The context size is approximately $t_{\text{summary}} + K \cdot \bar{t}$, regardless of conversation length. Key facts from earlier turns survive in the summary.

2. **Compression is lossy.** Specific numbers, dates, and technical details are at highest risk. Names and general themes survive well. The retention rate depends heavily on how aggressive the compression is.

3. **Incremental summarization is more efficient** than regenerating the full summary each time. It processes only the newly-old messages, reducing the summarizer's workload.

4. **The compression-retention curve shows diminishing returns.** Going from a 100-token to a 300-token summary dramatically improves retention. Going from 300 to 800 tokens adds much less.

5. **Summary memory requires an LLM call for summarization.** This adds latency and cost for the summarization step. In production, you must balance summarization frequency against these costs.


In [ ]:
#@title 🎧 Listen: Reflection Next
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_32_reflection_next.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Questions to Think About

1. What if the summary itself contains a hallucination? The summarizer might introduce facts that were never in the conversation. How would you verify summary accuracy?

2. Could you combine summary memory with the sliding window — keeping a summary AND a relevance-based retrieval of old messages? What would that architecture look like?

3. Our summarizer treats all information equally. But in a customer support context, the user's order number is far more important than their greeting. How would you tell the summarizer what to prioritize?

4. What happens when the conversation shifts topics dramatically? A summary of a cooking discussion is useless when the user starts asking about programming. Should summaries be topic-aware?

5. How would you handle the case where the user explicitly corrects something? ("Actually, the budget is $8,000, not $5,000.") Should the summary be updated to reflect the correction?

### What Comes Next

Summary memory preserves general facts but loses specific details. In the next notebook, we will explore **entity memory** — a system that extracts specific entities (names, numbers, dates, preferences) and stores them in a structured dictionary. This gives us the best of both worlds: the compression of summaries and the precision of exact facts.

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_33_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
print("Notebook complete!")
print()
print("Key equations:")
print("  Summary cost: C = t_summary + K * t_avg (bounded)")
print("  Compression:  rho = t_summary / t_old (typically 3-10%)")
print("  Retention:    R = facts_found / total_facts")
print()
print("The key insight: summaries preserve themes but lose details.")
print("Next up: Entity Memory — structured fact extraction.")